# Earnings Yield (EYLD) — Risk-Factor Construction Spec (USE4-faithful)

**Purpose.** This notebook is the **source-of-truth spec** for building the
**EYLD** style factor following MSCI's description in *Barra USE4 Empirical Notes*
(Appendix A, p. 52; Section 3.4, p. 16). It is **not** a research notebook. The
deliverable is a clean, USE4-faithful Earnings Yield factor written to parquet,
suitable for input to a multi-factor risk model.

**Audience.** You — plus whatever AI assistant you are driving. Treat its output as deeply untrustworthy until audited. Follow the stages linearly. Each stage has:
- ✅ **PDF SPEC** — exact verbatim quote or paraphrase from the USE4 documentation
- ❓ **NOT IN PDF** — something the PDF does not specify; a judgment call you must make
- 💡 **DEFAULT** — a reasonable default for any ❓ item, chosen to match standard Barra practice
- 🧪 **VALIDATE** — sanity checks to run after each stage

**Inputs:**
- `SHARADAR_SF1.csv` — Sharadar SF1 fundamental data (dimension=TTM, PIT via `datekey`)
- `data/out/estu_monthly.parquet` — Estimation universe per signal_date (`in_estu`, `mcap`, monthly signal dates)

**Deliverable:** A parquet file `eyld_use4.parquet` keyed by `(permaticker, signal_date)`
containing the standardised EYLD exposure for every stock in the coverage universe on every
monthly signal date.

**Companion specs:**
- `02_style_factors/beta/beta_spec.ipynb` — Beta (upstream for NLB, ResVol)
- `02_style_factors/mom/mom_spec.ipynb` — Momentum (standalone)

## §1. The USE4 EYLD spec — verbatim quotes

### 1a. Barra USE4 Empirical Notes, Appendix A (p. 52) — formal descriptor definition

> **Earnings Yield (EYLD)**
>
> *Definition:* `0.75 · EPFWD + 0.15 · CETOP + 0.10 · ETOP`
>
> *EPFWD — Predicted earnings-to-price ratio:* "Twelve-month forward-looking earnings
> divided by current market cap. Forward earnings are a weighted average of
> analyst-predicted EPS for the current and next fiscal years."
>
> *CETOP — Cash earnings-to-price ratio:* "Trailing 12-month cash earnings divided
> by current price."
>
> *ETOP — Trailing earnings-to-price ratio:* "Trailing 12-month earnings divided by
> current market cap. Trailing earnings = last reported fiscal-year earnings +
> (current interim figure − comparative interim figure from prior year)."

### 1b. Barra USE4 Empirical Notes, Section 3.4 (p. 16) — economic role

> "The Earnings Yield factor describes return differences based on a company's earnings
> relative to its price. The Earnings Yield factor is the most significant factor
> in explaining value-related return differences in U.S. equities."

### 1c. Barra USE4 Methodology Notes, Section 2.3 (p. 9) — standardisation rule

> "Descriptors are standardised to have a mean of 0 and a standard deviation of 1.
> ... μ_l is the cap-weighted mean of the descriptor (within the estimation universe),
> and σ_l is the equal-weighted standard deviation."

---

**That is all the PDFs say about EYLD construction specifically.** The following are
NOT covered:
- Which data vendor provides fundamental data
- How to handle missing EPFWD (no forward estimates available)
- Exact TTM construction method from quarterly filings
- PIT alignment (filing date vs. fiscal period end)
- Treatment of negative earnings
- Staleness cutoff for fundamental reports

## §2. End-to-end pipeline

```
┌─────────────────────────────────────────────────────────────────────┐
│  STAGE 0:  Setup, imports, paths                                    │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 1:  Load Sharadar SF1 fundamentals (TTM, PIT via datekey)   │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 2:  Build ESTU per signal_date                               │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 3:  (Skipped — no market benchmark needed for EYLD)         │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 4:  EYLD estimator (CETOP + ETOP → 0.6/0.4 composite)      │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 5:  Outlier treatment  (3σ default)                         │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 6:  Standardise  (cap-weighted mean=0, EW std=1)            │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 7:  Save deliverable                                         │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 8:  Validate                                                 │
└─────────────────────────────────────────────────────────────────────┘
```

**PIT discipline:** For any signal_date `t`, the only permissible data is records
with `datekey ≤ t` (Sharadar filing date). `calendardate` (fiscal period end) lags
behind `datekey` — using it instead would introduce look-ahead bias.

**No daily price data is used.** EYLD is purely fundamental (accounting ratios).
Market cap on signal_date comes from `estu_monthly.parquet`.

## §3. Output schema — what the worker delivers

**Raw descriptor column:** `eyld`
**Standardised score column:** `eyld_score`

**File:** `data/out/eyld_use4.parquet`

**Compression:** zstd, statistics=True

**Schema:**

| Column | Type | Description |
|---|---|---|
| `permaticker` | Int64 | Sharadar permanent ticker ID |
| `signal_date` | Date | End-of-month rebalance date (last trading day of month) |
| `in_estu` | Bool | Whether this stock was in the estimation universe on this date |
| `mcap` | Float64 | Market cap on signal_date (from estu_monthly) |
| `eyld` | Float64 | **Raw composite descriptor** = 0.6·CETOP + 0.4·ETOP (Stage 4 output) |
| `eyld_score` | Float64 | **Final EYLD exposure** — standardised cross-sectionally (the deliverable) |
| `n_obs` | UInt32 | 1 if a valid TTM fundamental observation was found within the lookback window; 0 otherwise |

**Coverage rules:**
- Compute `eyld` and `eyld_score` for **every stock with a valid TTM observation**
  within the `LOOKBACK_DAYS` window, not just ESTU members.
- Standardisation moments (mean, std) are computed using **ESTU members only**.
- Non-ESTU stocks get the *same* standardisation parameters applied so the final
  factor is comparable across the coverage universe.

**Note on n_obs:** For this factor, `n_obs` is binary (1 = has a valid report,
0 = stale or missing). It does not count historical observations as it would for
a rolling-window time-series factor.

## §4. STAGE 0 — Setup, imports, paths

Standard imports. Use polars, not pandas, throughout (project convention).

✅ **PDF SPEC — EYLD descriptor weights:**
> EYLD = 0.75 · EPFWD + 0.15 · CETOP + 0.10 · ETOP
>
> (Appendix A, p. 52)

❓ **NOT IN PDF — EPFWD omitted (item 1):** EPFWD requires analyst consensus forward EPS
estimates which are not available in the Sharadar SF1 database. The original weights
assume EPFWD contributes 75% of the composite.

💡 **DEFAULT:** Drop EPFWD entirely and renormalize remaining weights over CETOP and ETOP:

```
EYLD_W_CETOP = 0.60   # renormalized from 0.15 / (0.15 + 0.10)
EYLD_W_ETOP  = 0.40   # renormalized from 0.10 / (0.15 + 0.10)
```

❓ **NOT IN PDF — staleness cutoff (item 7):** The PDF does not specify when a
fundamental report is considered too old to use.

💡 **DEFAULT:** Discard any report with `datekey` more than 18 months (~548 calendar
days) before `signal_date`. U.S. companies must file within 60 days of fiscal year-end
(accelerated filers), so 18 months is a conservative upper bound that catches stragglers
and delistings.

```python
# ───── Parameters ─────────────────────────────────────────────────────────────
EYLD_W_CETOP    = 0.60   # renormalized CETOP weight (EPFWD not available)
EYLD_W_ETOP     = 0.40   # renormalized ETOP weight
LOOKBACK_DAYS   = 548    # ~18 months; max staleness for fundamental report
MIN_OBS         = 1      # require at least 1 valid TTM observation within lookback

# 💡 DEFAULT (NOT IN PDF) — calendar start
CALENDAR_START = date(1999, 1, 1)

# -- Factor type (read by /build-factor to select boilerplate template)
FACTOR_TYPE = "fundamental"   # time_series or fundamental
```

## §5. STAGE 1 — Load Sharadar SF1 fundamentals

✅ **PDF SPEC:** CETOP and ETOP both use trailing 12-month (TTM) figures:
> - CETOP: "Trailing 12-month cash earnings divided by current price."
> - ETOP: "Trailing 12-month earnings divided by current market cap."
>
> (Appendix A, p. 52)

❓ **NOT IN PDF — data source (item 2):** The PDF does not specify which data vendor
provides the fundamental data.

💡 **DEFAULT:** Sharadar SF1 (`SHARADAR_SF1.csv`), using
`dimension = "TTM"` (trailing twelve months). Sharadar computes TTM figures by
summing the four most recent quarterly reports — no manual TTM construction needed.

❓ **NOT IN PDF — PIT alignment (item 3):** Sharadar SF1 has two date columns:
- `calendardate` — the fiscal period end date (e.g., 2024-12-31)
- `datekey` — the date the data was released / made available

Using `calendardate` introduces look-ahead bias of ~45–90 days (the typical earnings
announcement lag). Sharadar `datekey` is the correct PIT date.

💡 **DEFAULT:** Filter `sf1` to only rows with `datekey ≤ signal_date` when looking
up the most recent fundamental observation.

❓ **NOT IN PDF — cash earnings definition (item 4):** "Cash earnings" is not a standard
GAAP line item. The common definition is net income + depreciation & amortization.

💡 **DEFAULT:** `cash_earnings = netinc + depamor` using Sharadar SF1 columns.
This is the most widely-used proxy for operating cash generation.

🧪 **VALIDATE:**
- `dimension == "TTM"` rows only in the loaded table
- `netinc` and `depamor` are non-null for the vast majority of rows (>90%)
- `datekey` ≤ `calendardate` + 120 days (filing within 4 months of period end — flag outliers)
- Date range covers full history from CALENDAR_START

## §6. STAGE 2 — Build ESTU per signal_date

**Identical to the shared USE4 build infrastructure** — no EYLD-specific decisions here.

✅ **PDF SPEC:** The estimation universe (ESTU) is used to compute standardisation
moments. Non-ESTU stocks are standardised using the same moments.

💡 **DEFAULT:** Load `estu_monthly.parquet` — prebuilt by `estu_build.ipynb`. This file
provides `(permaticker, signal_date, in_estu, mcap)` for every stock on every
end-of-month signal date.

**Key: `mcap` from `estu_monthly` is the signal_date market cap used as the denominator
for CETOP and ETOP.** Do not use the `marketcap` column from SF1, which is measured as of
the report filing date — not the signal date.

🧪 **VALIDATE:**
- ESTU has ~2,400–2,600 `in_estu = True` stocks per date (post-2005)
- Signal dates are the last trading day of each month
- `mcap` is non-null and positive for all `in_estu = True` rows

## §7. STAGE 3 — Market benchmark

**Not required for EYLD.** This factor is computed from fundamental accounting ratios;
no market return data is needed. Skip this stage entirely.

(Stage 3 computes a cap-weighted ESTU market return series, which is used by time-series
factors such as Beta, Momentum, NLB, and NLS to compute excess returns. EYLD does not
involve any return time series.)

## §8. STAGE 4 — EYLD estimator

✅ **PDF SPEC (Barra USE4 Empirical Notes, Appendix A, p. 52):**

> EYLD = 0.75 · EPFWD + 0.15 · CETOP + 0.10 · ETOP
>
> CETOP = (trailing 12-month cash earnings) / (current price)
>
> ETOP = (trailing 12-month earnings) / (current market cap)

❓ **NOT IN PDF — EPFWD omitted (item 1):** Forward earnings estimates unavailable.
Composite renormalized over CETOP and ETOP only.

💡 **DEFAULT:** EYLD = 0.60 · CETOP + 0.40 · ETOP

❓ **NOT IN PDF — mcap denominator for both sub-descriptors (item 5):** CETOP uses
"current price" while ETOP uses "current market cap." These are equivalent up to a
shares-outstanding scaling factor: price = mcap / shares.

💡 **DEFAULT:** Use `mcap` from `estu_monthly` (signal-date market cap) as the
denominator for **both** CETOP and ETOP. This ensures the ratio is dimensionally
consistent (total earnings / total market cap = aggregate earnings yield). Per-share
normalisation cancels when both numerator and denominator are per-share — using total
figures is equivalent and avoids needing a shares outstanding column.

❓ **NOT IN PDF — negative earnings (item 6):** Stocks with negative net income produce
a negative earnings yield. The USE4 PDF does not specify whether these should be
clipped, excluded, or left as-is.

💡 **DEFAULT:** Allow negative earnings yields. Negative E/P stocks (value traps) are
genuinely informative — they should load negatively on the factor. The 3σ outlier trim
in Stage 5 handles extreme cases (e.g., massive one-time charges).

❓ **NOT IN PDF — "most recent" report selection (item 7):** For each stock on each
signal_date, multiple TTM rows may be available (one per quarter). We need exactly one.

💡 **DEFAULT:** Select the row with the maximum `datekey ≤ signal_date`, subject to
`datekey > signal_date - LOOKBACK_DAYS`. This is the most recently filed TTM report
available PIT. If no row exists within the lookback window, the stock gets `eyld = null`.

---
*The section above (PDF SPEC quote through final 💡 DEFAULT) is the `STAGE4_DESCRIPTION`
that `/build-factor` will inject verbatim into the Stage 4 stub docstring. Content below
this line is supplementary guidance for the implementer and is not extracted.*
---

**Implementation notes:**
- Sort `sf1_ttm` by `(permaticker, datekey)` and use `join_asof` with `strategy="backward"`
  on `datekey` to efficiently find the most recent report per stock per signal_date.
- After the join, apply the staleness filter: drop rows where
  `signal_date - datekey > LOOKBACK_DAYS` calendar days.
- Compute sub-descriptors then composite:

```
cetop_i = (netinc_i + depamor_i) / mcap_i
etop_i  = netinc_i / mcap_i
eyld_i  = EYLD_W_CETOP * cetop_i + EYLD_W_ETOP * etop_i
```

- Guard: filter `mcap > 0` before division (rare Sharadar data errors).
- `n_obs = 1` for every stock that passes the lookback filter.

🧪 **VALIDATE:**
- Raw `eyld` cross-sectional range: most stocks fall in roughly −0.15 to +0.30
  (realistic earnings yields for U.S. equities)
- Median `eyld` (ESTU) should be modestly positive (~0.04–0.08 for typical market)
- High E/P value stocks (energy, mature industrials) → highest `eyld`
- High-multiple growth / loss-making stocks (unprofitable tech, biotech) → lowest `eyld`
- Coverage ≥ 4,000 stocks per date post-2005

## §9. STAGE 5 — Outlier treatment

❓ **NOT IN PDF for EYLD specifically.** Methodology Notes Section 2.2 (p. 8)
describes a general outlier-treatment algorithm that applies to all descriptors:

> *"We trim these observations to three standard deviations from the mean."*

💡 **DEFAULT:** Apply 3σ trimming per signal_date, computed within ESTU using
cap-weighted mean ± 3 × equal-weighted std. Applied to all stocks in the coverage
universe.

**Note:** EYLD can have heavy right-skew for distressed stocks (enormous negative mcap
growth reduces the denominator) and left-skew for cyclicals at trough earnings. The 3σ
trim is particularly important for this factor — without it, a handful of tiny-cap
turnaround stocks with extremely high E/P can dominate the distribution.

🧪 **VALIDATE:**
- ~0.5–2% of ESTU rows should be clipped at either bound per date
- After trimming, cross-sectional skewness and kurtosis of `eyld` should be lower
- Clipped rows should be concentrated in the positive tail (tiny-cap high E/P) and
  the negative tail (loss-making stocks with large write-downs)

## §10. STAGE 6 — Standardise (cap-weighted mean=0, EW std=1)

✅ **PDF SPEC (Methodology Notes §2.3, p. 9):**

> *"μ_l is the cap-weighted mean of the descriptor (within the estimation universe),
> and σ_l is the equal-weighted standard deviation."*

**Per signal_date `t`, using only ESTU members:**

```
μ_t  = Σ_{i ∈ ESTU_t} (mcap_i · eyld_i) / Σ mcap_i   # cap-weighted mean
σ_t  = std_{i ∈ ESTU_t}(eyld_i)                         # equal-weighted std
eyld_score_{i,t} = (eyld_i − μ_t) / σ_t                # applied to ALL i
```

Three critical points:
1. Moments estimated on **ESTU only**; applied to **every stock** in the coverage universe.
2. Mean is **cap-weighted**; std is **equal-weighted**.
3. Cap-weighting the mean ensures the cap-weighted market portfolio has zero EYLD exposure.

🧪 **VALIDATE:**
- `Σ_{i ∈ ESTU} (mcap_i · eyld_score_i) / Σ mcap_i ≈ 0` (cap-weighted mean = 0 by construction)
- `std_{i ∈ ESTU}(eyld_score_i) ≈ 1` (equal-weighted std = 1 by construction)

## §11. STAGE 7 — Save deliverable

Write the final panel to `data/out/eyld_use4.parquet`.
Compression: zstd. Statistics: True.

Include `eyld` (raw composite) alongside `eyld_score` (standardised) for downstream
auditing and factor-model input. `n_obs` is included for coverage diagnostics.

The downstream consumer (factor model, audit notebook) will use `eyld_score`.

## §12. STAGE 8 — Validation

### Required checks (all must pass before signing off)

**1. Standardisation moments on ESTU:**
```
cap-weighted mean of eyld_score ≈ 0   (|mean| < 1e-6)
equal-weighted std of eyld_score ≈ 1   (|std − 1| < 0.02)
```

**2. Raw descriptor sanity:**
```
Median eyld (ESTU) in range [0.01, 0.12]  (typical U.S. market E/P range)
Max |eyld| post-trim < 1.0               (no stock has E/P > 100% after 3σ clip)
```

**3. Coverage stability:**
```
≥ 4,000 stocks with non-null eyld_score per date post-2005
```

**4. Factor stability (month-over-month rank correlation):**
```
MoM Spearman ρ > 0.85  (fundamentals are sticky; EYLD is more stable than MOM)
```

**5. Economic direction:**
```
Q5 (high EYLD) contains value-oriented stocks with high E/P ratios.
Q1 (low EYLD) contains high-multiple growth stocks or loss-making companies.
Spot check: energy majors and mature industrials → Q5.
Spot check: unprofitable tech/biotech → Q1.
```

**6. Disk vs memory equivalence:**
```
max abs diff of eyld values after read-back < 1e-10
```

---

**Shared validation conventions (all factors, 2026-06-10):**
- **Coverage (check 3)** is evaluated on **completed months only** — the final
  signal date is the in-progress month (freshest data can lag the Sharadar
  refresh) and is excluded from the floor.
- **Fraction of scores in [−3, +3]** is reported for the full universe *and* for
  ESTU; the ESTU figure is the operationally relevant one (non-ESTU micro-caps
  pull the all-universe tail).
- Common check machinery lives in `common/ (your shared utilities)`
  .

## §13. Master list of ❓ NOT-IN-PDF decisions

| # | Decision | Default chosen | Alternative | Where to revisit |
|---|---|---|---|---|
| 1 | EPFWD omitted — no forward estimates in database | Renormalize to 0.6·CETOP + 0.4·ETOP | Proxy EPFWD with trailing EPS (same as ETOP) | If forward estimate data becomes available |
| 2 | Data source for fundamentals | Sharadar SF1 ARQ; TTM constructed as 4-quarter rolling sums | Any other fundamental data vendor | Never (tied to the Sharadar data stack) |
| 3 | PIT alignment | Use `datekey` (filing date) as PIT constraint | Use `calendardate` + fixed lag (e.g. +60 days) | If Sharadar datekey quality is suspected |
| 4 | Cash earnings definition | netinc + depamor | FCF, EBITDA, or operating cash flow from CF statement | If different definition matches USE4 intent better |
| 5 | mcap denominator source | estu_monthly mcap on signal_date | SF1 marketcap on report date | Never — signal_date mcap is always correct for E/P ratios |
| 6 | Negative earnings treatment | Allow; 3σ trim handles extremes | Floor at 0; exclude loss-makers | If factor is used in a value-only context |
| 7 | Staleness cutoff | 548 calendar days (~18 months) | 365 days (~1 year, more conservative) | If stale fundamentals are suspected to add noise |
| 8 | Financial sector treatment | No special treatment; include all sectors | Separate sector-neutral standardisation | If financial sector distorts the factor |
| 9 | MIN_OBS threshold | 1 (binary: has valid TTM or not) | Higher if TTM quality check required | If n_obs proves uninformative |
| 10 | Calendar start | 1999-01-01 | 2000-01-01 (avoid thin Sharadar pre-2000 coverage) | If early data quality is poor |

## §14. Common pitfalls — read this before coding

**Pitfall 1: Using calendardate instead of datekey for PIT**
Sharadar's `calendardate` is the end of the fiscal period (e.g., Dec 31). Earnings are
not *reported* until 30–90 days later (`datekey`). Using `calendardate ≤ signal_date`
as the filter introduces 1–3 months of look-ahead bias on every signal date — a
systematic form of data snooping that inflates backtest returns.

**Pitfall 2: Using dimension other than TTM**
SF1 has multiple dimension codes: MRQ, MRY, ARQ, ARY, TTM. Only `dimension = "TTM"`
gives trailing 12-month totals directly. Using MRQ (most recent quarter) without
annualising would understate the ratio by ~4×. Using ARY (annual as-reported) misses
interim periods and introduces quarterly seasonality.

**Pitfall 3: Denominator mismatch — SF1 marketcap vs estu_monthly mcap**
SF1's `marketcap` field reflects the market cap as of the `datekey` (report filing date).
For E/P ratios, the denominator should always be the market cap on the *signal date*
(when the factor is evaluated). Using `sf1.marketcap` shifts the denominator by 1–4
months, causing stale price signals.

**Pitfall 4: Not filtering mcap ≤ 0 before division**
Rare but real: some Sharadar records have `marketcap = 0` or negative mcap (data errors).
Always filter `mcap > 0` from `estu_monthly` before computing the ratio to avoid
division-by-zero or sign-flip in CETOP/ETOP.

**Pitfall 5: Missing depamor vs zero depamor**
Some companies (financials, holding companies) legitimately report zero D&A. Other rows
in SF1 have null `depamor` (data not available). Treat `null` as missing data (exclude
from CETOP) — but treat `0.0` as valid (cash earnings = net income for that company).
Do not `fill_null(0.0)` across the board; check whether nulls are structural or gaps.

## §15. Final summary — what "done" looks like

**The deliverable is one file:** `data/out/eyld_use4.parquet`

**Acceptance criteria:**

1. ✅ Schema matches §3
2. ✅ All 6 required validation checks in §12 pass within tolerance
3. ✅ ESTU has ~2,400–2,600 names per date post-2005, stable over time
4. ✅ Cap-weighted mean of `eyld_score` is machine-zero for every date in ESTU
5. ✅ Equal-weighted std of `eyld_score` is 1.0 (to 2 decimal places) for every date in ESTU
6. ✅ Median raw `eyld` (ESTU) is in [0.01, 0.12] for every post-2005 date
7. ✅ Coverage of `eyld_score` across coverage universe ≥ 4,000 stocks per date post-2005
8. ✅ Month-over-month rank stability ρ > 0.85
9. ✅ All 10 ❓ NOT-IN-PDF decisions in §13 are documented in the build notebook code

**Once EYLD is built and passes validation, it is a standalone factor** — no other
USE4 factors depend on EYLD as an upstream input. It feeds directly into the
multi-factor risk model as one of the style factors.